# Notebook 03 — ERD Implementation, DML Ingestion, and Master Practice Challenges

**Session:** Database - SQL (2)  
**Module:** 02 · Calculus and Linear Algebra

## Goal
1. Implement relational schemas from scratch in proper topological order with referential integrity constraints.
2. Validate constraint enforcement (foreign key rejection, composite key uniqueness).
3. Master DML ingestion patterns (positional, explicit-column, and multi-row bulk batching).
4. Execute and verify all 10 master analytical SQL practice problems from the session study guide with automated assertions.

In [ ]:
import sqlite3
from pathlib import Path


def run_ddl_and_verify() -> sqlite3.Connection:
    """Create a new in-memory SQLite database and test foreign key constraints."""
    c = sqlite3.connect(":memory:")
    c.execute("PRAGMA foreign_keys = ON;")
    seed_path = Path("../data/sql_store_seed.sql")
    c.executescript(seed_path.read_text(encoding="utf-8"))
    return c


conn = run_ddl_and_verify()


def query(sql: str) -> list[dict]:
    cur = conn.execute(sql)
    cols = [desc[0] for desc in cur.description]
    return [dict(zip(cols, row, strict=True)) for row in cur.fetchall()]


def show_rows(rows: list[dict], limit: int | None = None) -> None:
    display_rows = rows if limit is None else rows[:limit]
    if not display_rows:
        print("(0 rows)")
        return
    cols = list(display_rows[0].keys())
    widths = {c: max(len(c), *(len(str(r[c])) for r in display_rows)) for c in cols}
    header = " | ".join(f"{c:<{widths[c]}}" for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)
    for r in display_rows:
        print(" | ".join(f"{r[c]!s:<{widths[c]}}" for c in cols))


print("Database initialized with foreign keys enabled.")

## 1. Validating Referential Integrity and Constraints

Foreign key constraints protect against orphan records. Attempting to insert an order with an invalid `customer_id` raises `sqlite3.IntegrityError`.

In [ ]:
# Test 1: Foreign key violation check
foreign_key_violation_caught = False
try:
    conn.execute(
        "INSERT INTO orders (order_id, customer_id, order_date, status) VALUES (999, 99999, '2026-11-20', 1);"
    )
except sqlite3.IntegrityError as e:
    foreign_key_violation_caught = True
    print(f"Successfully caught expected foreign key violation: {e}")

assert foreign_key_violation_caught

# Test 2: Composite Primary Key uniqueness on order_items
composite_key_violation_caught = False
try:
    # (order_id=1, product_id=4) already exists
    conn.execute(
        "INSERT INTO order_items (order_id, product_id, quantity, unit_price) VALUES (1, 4, 10, 5.00);"
    )
except sqlite3.IntegrityError as e:
    composite_key_violation_caught = True
    print(f"Successfully caught expected composite primary key collision: {e}")

assert composite_key_violation_caught

## 2. DML Ingestion Patterns

We test three DML insertion strategies: positional, column-explicit, and multi-row bulk batching.

In [ ]:
# 1. Positional Insert into shippers
conn.execute("INSERT INTO shippers VALUES (6, 'Global Air Cargo');")
s6 = query("SELECT * FROM shippers WHERE shipper_id = 6;")[0]
assert s6["name"] == "Global Air Cargo"

# 2. Explicit Column Insert into customers
conn.execute("""
INSERT INTO customers (customer_id, first_name, last_name, birth_date, address, city, state, points)
VALUES (13, 'Cassandra', 'Phelps', '1998-04-12', '104 Beacon Hill', 'Boston', 'MA', 2150);
""")
c13 = query("SELECT * FROM customers WHERE customer_id = 13;")[0]
assert c13["first_name"] == "Cassandra"
assert c13["phone"] is None

# 3. Multi-Row Bulk Insert into products
conn.execute("""
INSERT INTO products (product_id, name, quantity_in_stock, unit_price) VALUES
    (14, 'Matcha Green Tea', 45, 14.50),
    (15, 'Ceramic Teapot', 25, 28.00),
    (16, 'Bamboo Whisk', 80, 8.25);
""")
new_prods = query("SELECT COUNT(*) AS c FROM products WHERE product_id >= 14;")[0]["c"]
assert new_prods == 3
print("DML insertion patterns validated successfully!")

## 3. The 10 Master SQL Practice Challenges

Here we execute all 10 analytical challenges from the study guide and assert mathematical correctness.

In [ ]:
# Challenge 1: Orders placed in 2018 with customer names
ch1 = query("""
SELECT o.order_id, o.order_date, c.first_name, c.last_name
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_date BETWEEN '2018-01-01' AND '2018-12-31'
ORDER BY o.order_id;
""")
assert len(ch1) == 5
assert [r["order_id"] for r in ch1] == [2, 6, 7, 8, 10]
print("Challenge 1: PASS")

# Challenge 2: Order #2 computed line-item values
ch2 = query("""
SELECT oi.product_id, p.name, oi.quantity, oi.unit_price, ROUND(oi.quantity * oi.unit_price, 2) AS item_total
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
WHERE oi.order_id = 2
ORDER BY item_total DESC;
""")
assert len(ch2) == 3
assert ch2[0]["item_total"] == 18.20
print("Challenge 2: PASS")

# Challenge 3: Staff hierarchy self-join
ch3 = query("""
SELECT
    e.first_name || ' ' || e.last_name AS employee_name,
    m.first_name || ' ' || m.last_name AS manager_name
FROM employees e
JOIN employees m ON e.reports_to = m.employee_id
ORDER BY employee_name;
""")
assert len(ch3) == 7
print("Challenge 3: PASS")

# Challenge 4: Orders with non-delivered status labels
ch4 = query("""
SELECT o.order_id, c.last_name, s.name AS status
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_statuses s ON o.status = s.order_status_id
WHERE s.name != 'Delivered'
ORDER BY o.order_id;
""")
assert len(ch4) == 14
print("Challenge 4: PASS")

# Challenge 5: Invoicing 4-way payment audit
ch5 = query("""
SELECT p.payment_id, c.name AS client_name, i.number AS invoice_num, pm.name AS method
FROM payments p
JOIN clients c ON p.client_id = c.client_id
JOIN invoices i ON p.invoice_id = i.invoice_id
JOIN payment_methods pm ON p.payment_method = pm.payment_method_id
ORDER BY p.payment_id;
""")
assert len(ch5) == 8
print("Challenge 5: PASS")

# Challenge 6: Annotated packaging line items
ch6 = query("""
SELECT oi.order_id, p.name, oin.note
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN order_item_notes oin ON oi.order_id = oin.order_id AND oi.product_id = oin.product_id
ORDER BY oi.order_id, oi.product_id;
""")
assert len(ch6) == 8
print("Challenge 6: PASS")

# Challenge 7: Customer left join order auditing
ch7 = query("""
SELECT c.customer_id, o.order_id
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE c.customer_id <= 12
ORDER BY c.customer_id, o.order_id;
""")
assert len(ch7) == 19
print("Challenge 7: PASS")

# Challenge 8: Multi-tier outer join with carriers
ch8 = query("""
SELECT c.first_name, o.order_id, sh.name AS carrier
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
LEFT JOIN shippers sh ON o.shipper_id = sh.shipper_id
WHERE c.customer_id <= 12
ORDER BY c.customer_id, o.order_id;
""")
assert len(ch8) == 19
print("Challenge 8: PASS")

# Challenge 9: Anti-join for unsold inventory
ch9 = query("""
SELECT p.product_id, p.name
FROM products p
LEFT JOIN order_items oi ON p.product_id = oi.product_id
WHERE oi.order_id IS NULL
ORDER BY p.product_id;
""")
# Note: products 7, 12, 14, 15, 16 are unsold
assert len(ch9) == 5
assert ch9[0]["product_id"] == 7
assert ch9[1]["product_id"] == 12
print("Challenge 9: PASS")

# Challenge 10: Customer loyalty tier union segmentation
ch10 = query("""
SELECT customer_id, points, 'Bronze' AS tier FROM customers WHERE points < 1500
UNION
SELECT customer_id, points, 'Silver' AS tier FROM customers WHERE points BETWEEN 1500 AND 2500
UNION
SELECT customer_id, points, 'Gold' AS tier FROM customers WHERE points > 2500
ORDER BY points DESC;
""")
assert len(ch10) == 13
print("Challenge 10: PASS")
print("ALL 10 MASTER SQL PRACTICE CHALLENGES PASSED!")